# 2. Screenshots, population, order, pair correlation and thesis figures

In [ ]:
from pathlib import Path
import numpy as np

from bulk_analysis import (
    average_gr,
    load_run,
    make_two_phase_figure,
    plot_population_density_order,
    plot_run_summary,
    second_peak,
    set_plot_style,
)

set_plot_style(11)

T_MIN = 1000

MAIN_DATA_FOLDER = Path("data/params_four_phases")
MAIN_PARAM_BASE = "params_four_phases"

# Existing old alternative data:
ALT_DATA_FOLDER = Path("data/params_four_phases_altered")
ALT_PARAM_BASE = "params_four_phases_altered"

FIGURE_FOLDER = Path("figures/bulk_validation")
THESIS_FOLDER = FIGURE_FOLDER / "thesis"

FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)
THESIS_FOLDER.mkdir(parents=True, exist_ok=True)

## Check the four phase measurements

In [ ]:
phase_names = {
    0: "Periodic clustered",
    1: "Spatially structured flocking",
    2: "Flocking",
    3: "Disordered",
}

for run_id, expected_phase in phase_names.items():
    run = load_run(
        MAIN_DATA_FOLDER,
        MAIN_PARAM_BASE,
        run_id,
    )

    r, g_mean = average_gr(run["gr_frames"], T_MIN)
    peak_r, peak_g = second_peak(r, g_mean)

    mask = run["S_data"]["time"] >= T_MIN
    mean_S = np.mean(run["S_data"]["S"][mask])

    spatial_order = peak_g > 1.2
    polar_order = mean_S > 0.7

    if spatial_order and polar_order:
        measured_phase = "Spatially structured flocking"
    elif spatial_order:
        measured_phase = "Periodic clustered"
    elif polar_order:
        measured_phase = "Flocking"
    else:
        measured_phase = "Disordered"

    print(
        f"run {run_id}: "
        f"expected = {expected_phase}, "
        f"measured = {measured_phase}, "
        f"second peak = {peak_g:.3f}, "
        f"mean S = {mean_S:.3f}"
    )

## One summary figure for each run

In [ ]:
for run_id, phase_name in phase_names.items():
    plot_run_summary(
        data_folder=MAIN_DATA_FOLDER,
        param_base=MAIN_PARAM_BASE,
        run_id=run_id,
        title=phase_name,
        t_min=T_MIN,
        save_path=FIGURE_FOLDER / f"run_{run_id:03d}_summary.pdf",
    )

## Population, density and polar order

In [ ]:
SELECTED_RUN_ID = 0

selected_run = load_run(
    MAIN_DATA_FOLDER,
    MAIN_PARAM_BASE,
    SELECTED_RUN_ID,
)

L = float(selected_run["params"]["L"])

plot_population_density_order(
    selected_run["S_data"],
    L,
    save_path=FIGURE_FOLDER / f"run_{SELECTED_RUN_ID:03d}_population_density_order.pdf",
)

## Final thesis Figures

In [ ]:
set_plot_style(15)

periodic_clustered = {
    "name": "Periodic clustered phase",
    "data_folder": MAIN_DATA_FOLDER,
    "param_base": MAIN_PARAM_BASE,
    "run_id": 0,
}

structured_flocking = {
    "name": "Spatially structured flocking phase",
    "data_folder": MAIN_DATA_FOLDER,
    "param_base": MAIN_PARAM_BASE,
    "run_id": 1,
}

flocking = {
    "name": "Flocking phase",
    "data_folder": MAIN_DATA_FOLDER,
    "param_base": MAIN_PARAM_BASE,
    "run_id": 2,
}

disordered = {
    "name": "Disordered phase",
    "data_folder": MAIN_DATA_FOLDER,
    "param_base": MAIN_PARAM_BASE,
    "run_id": 3,
}

make_two_phase_figure(
    [periodic_clustered, structured_flocking],
    THESIS_FOLDER / "figure_4_1_spatial_order",
    T_MIN,
)

make_two_phase_figure(
    [disordered, flocking],
    THESIS_FOLDER / "figure_4_2_no_spatial_order",
    T_MIN,
)

In [ ]:
original_inheritance = {
    "name": "Original inheritance rule",
    "data_folder": MAIN_DATA_FOLDER,
    "param_base": MAIN_PARAM_BASE,
    "run_id": 1,
}

random_orientation = {
    "name": "Modified inheritance rule",
    "data_folder": ALT_DATA_FOLDER,
    "param_base": ALT_PARAM_BASE,
    "run_id": 1,
}

make_two_phase_figure(
    [original_inheritance, random_orientation],
    THESIS_FOLDER / "figure_4_3_directional_inheritance",
    T_MIN,
)